# 03. Evaluasi, output generasi, dan keputusan tahap berikutnya

Tujuan: membaca bukti hasil model dan memahami pekerjaan yang masih diperlukan.

**Dua ukuran berbeda:** loss/perplexity mengukur prediksi token kode; parse/schema mengukur bentuk PageSpec yang diperlukan produk.
Empat prompt saat ini merupakan probe eksploratif yang belum direview, sehingga bukan benchmark SFT produksi.
Notebook membaca hasil tersimpan. Tidak melatih atau memilih checkpoint berdasarkan test.

In [1]:
from pathlib import Path
import sys, json, shutil, datetime
import pandas as pd
import torch
from IPython.display import display, Markdown, Image

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "configs/model.json").is_file() and (p / "src/models/minigpt.py").is_file()), None)
if ROOT is None:
    raise RuntimeError("Buka notebook dari folder proyek ini.")
if Path(sys.prefix).resolve() != (ROOT / ".venv").resolve():
    raise RuntimeError("Pilih kernel Mini GPT WebCodeGen dari .venv proyek ini.")
sys.path.insert(0, str(ROOT))
def load_json(relative):
    return json.loads((ROOT / relative).read_text(encoding="utf-8"))
print("Interpreter:", sys.executable)
print("Waktu eksekusi:", datetime.datetime.now(datetime.timezone(datetime.timedelta(hours=7))).isoformat())
print("PyTorch:", torch.__version__, "| CUDA:", torch.version.cuda)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

Interpreter: F:\Generative_AI\Vibe_Coding_Proyek_1\.venv\Scripts\python.exe
Waktu eksekusi: 2026-10-12T22:17:11.039303+07:00
PyTorch: 2.14.1+cu130 | CUDA: 13.0
GPU: NVIDIA GeForce RTX 5070


## 1. Identitas checkpoint dan test partition

Checkpoint terbaik dipilih menggunakan validation. Setelah itu, RUN-066 mengevaluasi seluruh test token.
Jumlah label valid adalah jumlah token cache dikurangi satu karena prediksi next-token.

In [2]:
report = load_json("reports/training/TRAIN-004_evaluation.json")
print("Checkpoint:", report["checkpoint"])
print("Step:", report["checkpoint_step"])
print("SHA256:", report["checkpoint_sha256"])
print("Partition:", report["held_out_token_partition"])
token_metrics = report["held_out_token_metrics"]
display(pd.DataFrame([token_metrics]))
import math
print("exp(loss):", math.exp(token_metrics["loss"]))
print("Gerbang aplikasi:", report["application_gate"])

Checkpoint: checkpoints\TRAIN-004\best.pt
Step: 1200
SHA256: 9f407be207c9b9090104d48243109a2c333d9609cd89e7584dd8ccd5d5b50486
Partition: test


,loss,perplexity,valid_tokens,batches
0,1.070987,2.918259,2074051,507


exp(loss): 2.9182586860057063
Gerbang aplikasi: needs-data


## 2. Parsing JSON dan validasi schema

Evaluator terlebih dahulu melakukan JSON parsing, lalu memeriksa schema dan aturan semantik PageSpec.
Raw output dan alasan gagal tetap dicatat. Tidak ada skor visual/layout otomatis atau eksekusi browser pada evaluasi ini.

In [3]:
generation = report["generation"]
display(pd.DataFrame([{k: generation[k] for k in ["count", "parseable", "schema_valid", "parseable_rate", "schema_valid_rate", "generation_mode", "browser_evaluated", "manual_review_completed"]}]))
records = generation["records"]
display(pd.DataFrame([{ "id": r["prompt_id"], "parse": r["parse_status"], "stop": r["stop_reason"],
                       "tokens": r["generated_tokens"], "schema_errors": len(r["schema_errors"]),
                       "review": r.get("review_status")} for r in records]))

,count,parseable,schema_valid,parseable_rate,schema_valid_rate,generation_mode,browser_evaluated,manual_review_completed
0,4,0,0,0.0,0.0,exploratory_unreviewed,False,False


,id,parse,stop,tokens,schema_errors,review
0,probe-001,failed,eos,17,0,not_reviewed
1,probe-002,failed,eos,154,0,not_reviewed
2,probe-003,failed,token_limit,512,0,not_reviewed
3,probe-004,failed,eos,166,0,not_reviewed


## 3. Bandingkan prompt dengan keluaran aktual

Semua prompt dan raw output ditampilkan sebagai teks. Perhatikan bahwa baseline masih melanjutkan fragmen HTML,
belum menyusun JSON berbahasa Indonesia yang sesuai prompt. Hasil yang gagal tetap menjadi bukti eksperimen.

In [4]:
for record in records:
    print("\n" + "="*65)
    print(record["prompt_id"], "|", record["prompt"])
    print("RAW OUTPUT:\n", record["raw_output"])
    print("Parse error:", record["parse_error"])


probe-001 | Buat landing page studio desain dengan hero, tiga layanan, dan footer. Gunakan CSS biasa, warna biru, serta teks bahasa Indonesia.
RAW OUTPUT:
 .
        </a>
    </div>
</body>
</html>
Parse error: Expecting value: line 1 column 1 (char 0)

probe-002 | Buat halaman katalog buku memakai Tailwind dengan empat kartu buku dan pencarian dari katalog interaksi filter.
RAW OUTPUT:
 .
        </div>
        <div class="flex flex-col md:flex-row">
            <div class="w-full md:w-1/2 p-4">
                <h2 class="text-2xl font-bold mb-4">Our Services</h2>
                <p class="text-lg">
                    We are a wide range of services to meet your needs. Our team of experienced professionals is dedicated to providing our clients with the best possible service and we are committed to providing our clients. We believe in the best possible service and we are committed to providing our clients with the best possible service.
                </p>
            </div>
       

## 4. Jalankan validator pada contoh kecil

Fixture berikut menunjukkan kontrak aplikasi. JSON valid belum tentu sesuai prompt atau memiliki layout yang baik.
Review manusia tetap menilai kesesuaian prompt, teks, layout, dan interaksi.

In [5]:
from src.evaluation.generate import assess_output
valid_fixture = {"schema_version": "1.0", "title": "Halo", "language": "id", "style_mode": "plain_css",
                 "theme": {"primary": "#2563eb", "background": "#ffffff", "text": "#111827", "font": "sans"},
                 "sections": [{"id": "hero-1", "type": "hero", "props": {"heading": "Halo", "text": "Selamat datang."}}], "interactions": []}
for label, text in [("JSON valid", json.dumps(valid_fixture)), ("HTML", "<h1>Halo</h1>"), ("JSON tanpa field", "{}")]:
    result = assess_output(text)
    print(label, {k: v for k, v in result.items() if k != "parsed_spec"})

JSON valid {'parse_status': 'passed', 'parse_error': None, 'schema_errors': []}
HTML {'parse_status': 'failed', 'parse_error': 'Expecting value: line 1 column 1 (char 0)', 'schema_errors': []}
JSON tanpa field {'parse_status': 'passed', 'parse_error': None, 'schema_errors': ["$: 'schema_version' is a required property", "$: 'title' is a required property", "$: 'language' is a required property", "$: 'style_mode' is a required property", "$: 'theme' is a required property", "$: 'sections' is a required property", "$: 'interactions' is a required property"]}


## 5. Kesiapan dan pekerjaan berikutnya

| Bagian | Bukti | Keputusan |
| --- | --- | --- |
| Data kode dan tokenizer | 51.211 record unik, BPE train-only | Tersedia |
| Model dan loop | 14.186.496 parameter, 1.200 update, stop/resume GPU | Baseline tersedia |
| Prediksi token test | Loss 1,071; perplexity 2,918 | Kode dipelajari |
| Prompt-ke-PageSpec | 0/4 JSON valid pada probe eksploratif | Belum siap |
| Data instruksi | 119 draft, reviewed 0 | Kurasi diperlukan |
| Preview dan unduhan | Belum dibuat | Tahap aplikasi setelah kualitas ditinjau |

Urutan selanjutnya:

1. Tinjau prompt Indonesia dan PageSpec, sambil mempertahankan source/family partition.
2. Periksa distribusi section, plain CSS/Tailwind, interaksi, dan panjang sequence.
3. Jalankan preparation reviewed, lalu profile dan SFT menggunakan checkpoint proyek.
4. Pilih checkpoint dengan validation. Evaluasi reviewed test serta rubric manusia.
5. Setelah kualitas cukup, bangun validator, renderer, preview, dan unduhan sesuai scope.

Test partition yang sudah dievaluasi jangan dipakai untuk menyesuaikan hyperparameter.
Untuk eksperimen masa depan, kelompok perlu menetapkan kebijakan benchmark dan final hold-out yang belum dibuka.

## 6. Evaluasi ulang setelah lineage diperketat

RUN-077 mengevaluasi verified copy melalui gate corpus/checkpoint yang baru. Tidak ada perubahan bobot atau tuning dari test. Perbandingan berikut memeriksa laporan hasil, bukan menjalankan training atau generasi tambahan.

In [6]:
verified = load_json("reports/training/TRAIN-004-VERIFIED_evaluation.json")
display(pd.DataFrame([{"artifact": "Baseline asli", **report["held_out_token_metrics"]}, {"artifact": "Verified copy", **verified["held_out_token_metrics"]}]))
print("Metrik token identik:", report["held_out_token_metrics"] == verified["held_out_token_metrics"])
print("JSON/schema copy:", verified["generation"]["parseable"], verified["generation"]["schema_valid"], "dari", verified["generation"]["count"])

,artifact,loss,perplexity,valid_tokens,batches
0,Baseline asli,1.070987,2.918259,2074051,507
1,Verified copy,1.070987,2.918259,2074051,507


Metrik token identik: True
JSON/schema copy: 0 0 dari 4


## 7. Evaluasi ulang secara terpisah

Gunakan report ID baru untuk membedakan eksperimen. Command ini memakai GPU, tetapi tidak mengubah bobot.

```powershell
.\.venv\Scripts\python.exe qa/run_logged.py --label evaluate_again -- .\.venv\Scripts\python.exe -m qa.evaluate_minigpt --checkpoint checkpoints/TRAIN-004/best.pt --report-id TRAIN-004-recheck
```

Rujukan: [laporan training](../reports/PIPELINE_TRAINING_SUMMARY.md), [log training](../logs/LOG_TRAINING.md), [README proyek](../README.md).